# W2-D2: Root Cause Analysis Pipeline

Graph traversal + temporal scoring + keyword kNN retrieval. No API key required.


## 1. Load input and build service graph


In [1]:
import json
import os
import re
from collections import defaultdict, deque
from datetime import datetime

CLUSTER_FILE = "../d1/results/cluster_summary.json"
ALERTS_FILE = "dataset/alerts_sample.jsonl"
SERVICES_FILE = "dataset/services.json"
HISTORY_FILE = "dataset/incidents_history.json"
OUTPUT_FILE = "results/rca_output.json"

with open(CLUSTER_FILE, encoding="utf-8") as f:
    cluster_summary = json.load(f)
with open(SERVICES_FILE, encoding="utf-8") as f:
    graph_data = json.load(f)
with open(HISTORY_FILE, encoding="utf-8") as f:
    history_payload = json.load(f)
incident_history = history_payload.get("incidents", history_payload) if isinstance(history_payload, dict) else history_payload
with open(ALERTS_FILE, encoding="utf-8") as f:
    raw_alerts = [json.loads(line) for line in f if line.strip()]

service_names = {s["name"] for s in graph_data["services"]}
graph = defaultdict(set)
reverse_graph = defaultdict(set)
for name in service_names:
    graph[name]; reverse_graph[name]
for edge in graph_data["edges"]:
    src, dst = edge["from"], edge["to"]
    if src in service_names and dst in service_names:
        graph[src].add(dst)
        reverse_graph[dst].add(src)

print(f"Loaded {len(cluster_summary['clusters'])} clusters, {len(raw_alerts)} alerts, {len(incident_history)} historical incidents")
print(f"Graph contains {len(service_names)} services and {sum(len(v) for v in graph.values())} service-to-service edges")

Loaded 3 clusters, 20 alerts, 29 historical incidents
Graph contains 10 services and 10 service-to-service edges


## 2. RCA and retrieval functions


In [2]:
SEVERITY_RANK = {"info": 0, "warn": 1, "crit": 2}

def parse_ts(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00"))

def cluster_alerts(cluster):
    start, end = map(parse_ts, cluster["time_range"])
    services = set(cluster["services"])
    return [a for a in raw_alerts if a["service"] in services and start <= parse_ts(a["ts"]) <= end]

def active_ancestors(service, active):
    seen, queue = set(), deque([service])
    while queue:
        node = queue.popleft()
        for caller in reverse_graph.get(node, set()):
            if caller in active and caller not in seen:
                seen.add(caller); queue.append(caller)
    return seen

def graph_temporal_top_k(cluster, top_k=3):
    active = set(cluster["services"])
    alerts = cluster_alerts(cluster)
    start, end = map(parse_ts, cluster["time_range"])
    duration = max((end - start).total_seconds(), 1.0)
    first_seen, max_sev = {}, {}
    for alert in alerts:
        svc = alert["service"]
        first_seen[svc] = min(first_seen.get(svc, parse_ts(alert["ts"])), parse_ts(alert["ts"]))
        max_sev[svc] = max(max_sev.get(svc, 0), SEVERITY_RANK.get(alert["severity"], 0))
    impacts = {svc: len(active_ancestors(svc, active)) for svc in active}
    max_impact = max(max(impacts.values()), 1)
    scored = []
    for svc in active:
        impact = impacts[svc] / max_impact
        temporal = 1 - ((first_seen.get(svc, end) - start).total_seconds() / duration)
        terminal = 1.0 if not (graph.get(svc, set()) & active) else 0.0
        severity = max_sev.get(svc, 0) / 2
        score = 0.40 * impact + 0.35 * temporal + 0.15 * terminal + 0.10 * severity
        scored.append([svc, round(score, 2)])
    return sorted(scored, key=lambda x: (-x[1], x[0]))[:top_k]

def tokenize(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def keyword_similarity(cluster, incident, graph_root=None):
    alerts = cluster_alerts(cluster)
    alert_context = []
    for alert in alerts:
        alert_context.extend([alert.get("service", ""), alert.get("metric", ""), alert.get("severity", ""), alert.get("labels", {}).get("note", "")])
    cluster_text = " ".join(cluster["services"] + cluster.get("fingerprints", []) + alert_context)
    incident_text = " ".join(incident.get("services_involved", [])) + " " + incident.get("summary", "") + " " + incident.get("root_cause_class", "")
    left, right = tokenize(cluster_text), tokenize(incident_text)
    keyword_overlap = len(left & right) / len(left) if left and right else 0.0
    cluster_services, hist_services = set(cluster["services"]), set(incident.get("services_involved", []))
    service_overlap = len(cluster_services & hist_services) / max(min(len(cluster_services), len(hist_services)), 1)
    root_agreement = 1.0 if graph_root and incident.get("root_cause_service") == graph_root else 0.0
    return 0.50 * keyword_overlap + 0.25 * service_overlap + 0.25 * root_agreement

def retrieve_similar(cluster, top_k=3, graph_root=None):
    scored = [(incident, keyword_similarity(cluster, incident, graph_root)) for incident in incident_history]
    scored = sorted((x for x in scored if x[1] > 0), key=lambda x: (-x[1], x[0].get("id", x[0].get("incident_id", "unknown"))))
    return scored[:top_k]

def analyze_cluster(cluster):
    graph_top3 = graph_temporal_top_k(cluster, top_k=3)
    root_cause = graph_top3[0][0] if graph_top3 else "unknown-svc"
    similar = retrieve_similar(cluster, top_k=3, graph_root=root_cause)
    if similar:
        best, retrieval_score = similar[0]
        confidence = round(min(0.99, 0.65 * graph_top3[0][1] + 0.35 * retrieval_score), 2)
        return {
            "cluster_id": cluster["cluster_id"], "graph_top3": graph_top3, "root_cause": root_cause,
            "class": best.get("root_cause_class", "other"), "confidence": confidence,
            "actions": ([best.get("remediation")] if best.get("remediation") else best.get("remediation_actions", ["Investigate manually"])),
            "reasoning": f"{root_cause} ranked first from graph impact and early-alert timing; top historical match {best.get('id', best.get('incident_id', 'unknown'))} supplied the class and actions.",
            "similar_incidents": [item[0].get("id", item[0].get("incident_id", "unknown")) for item in similar], "method": "graph+keyword-knn"
        }
    return {
        "cluster_id": cluster["cluster_id"], "graph_top3": graph_top3, "root_cause": root_cause,
        "class": "other", "confidence": round(0.5 * graph_top3[0][1], 2) if graph_top3 else 0.0,
        "actions": ["Investigate manually"], "reasoning": "Retrieval returned no similar incidents; graph-only fallback used.",
        "similar_incidents": [], "method": "graph-only-fallback"
    }

print("Defined graph traversal, temporal scoring, keyword similarity, and kNN-style retrieval")

Defined graph traversal, temporal scoring, keyword similarity, and kNN-style retrieval


## 3. Analyze every cluster


In [3]:
analyzed_results = [analyze_cluster(cluster) for cluster in cluster_summary["clusters"]]
for result in analyzed_results:
    print(result["cluster_id"], "root_cause=", result["root_cause"], "class=", result["class"], "confidence=", result["confidence"])
    print("  graph_top3:", result["graph_top3"])
    print("  similar:", result["similar_incidents"])

c-000-000 root_cause= payment-svc class= connection_pool_exhaustion confidence= 0.86
  graph_top3: [['payment-svc', 1.0], ['notification-svc', 0.9], ['cart-svc', 0.87]]
  similar: ['INC-2025-11-08', 'INC-2026-01-04', 'INC-2025-09-05']
c-000-001 root_cause= recommender-svc class= batch_overlap confidence= 0.61
  graph_top3: [['recommender-svc', 0.55]]
  similar: ['INC-2026-03-07', 'INC-2025-08-02', 'INC-2025-10-28']
c-000-002 root_cause= search-svc class= n_plus_1 confidence= 0.61
  graph_top3: [['search-svc', 0.55]]
  similar: ['INC-2026-01-29', 'INC-2026-05-25', 'INC-2025-09-21']


## 4. Write required output


In [4]:
final_output = {"clusters_analyzed": len(analyzed_results), "results": analyzed_results}
os.makedirs("results", exist_ok=True)
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)
print(f"Written {OUTPUT_FILE}")
print(json.dumps({"clusters_analyzed": final_output["clusters_analyzed"], "main_root_cause": final_output["results"][0]["root_cause"], "main_confidence": final_output["results"][0]["confidence"]}, indent=2))

Written results/rca_output.json
{
  "clusters_analyzed": 3,
  "main_root_cause": "payment-svc",
  "main_confidence": 0.86
}


## 5. Validate schema


In [5]:
required_result_fields = {"cluster_id", "graph_top3", "root_cause", "class", "confidence", "actions", "reasoning", "similar_incidents", "method"}
errors = []
if final_output["clusters_analyzed"] != len(cluster_summary["clusters"]): errors.append("cluster count mismatch")
for result in final_output["results"]:
    missing = required_result_fields - set(result)
    if missing: errors.append(f"{result.get('cluster_id')} missing {sorted(missing)}")
    if not result["graph_top3"]: errors.append(f"{result['cluster_id']} has empty graph_top3")
    if not result["root_cause"] or not result["class"]: errors.append(f"{result['cluster_id']} missing RCA fields")
print("ALL ACCEPTANCE CHECKS PASSED" if not errors else "VALIDATION FAILED")
for error in errors: print("-", error)

ALL ACCEPTANCE CHECKS PASSED
